# Getting Started with Decider

Decider is a Python framework for building composable, config-driven decision pipelines on top of Polars DataFrames. You define plain Python functions that return `pl.Expr`, and Decider automatically wires their dependencies, compiles them into an optimised expression graph, and executes them in a single Polars pass. Pipelines can be saved to versioned JSON configs and hot-swapped at serving time without any code changes.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../../')

import polars as pl
from decider.modules.functional import generate_from_functions

print('Decider imported successfully')

## Your First Module

Define one or more plain functions that return `pl.Expr`. Each function name becomes an output column. Parameter names map to input columns (or to sibling function outputs — more on that shortly).

In [ ]:
# --- define functions -------------------------------------------------------
# Each function: name -> output column, params -> input columns

def score(amount: pl.Expr, rate: pl.Expr) -> pl.Expr:
    """Raw score: amount adjusted by rate."""
    return amount * rate

def risk_flag(score: pl.Expr) -> pl.Expr:
    """Flag rows where score exceeds 80."""
    return score > pl.lit(80.0)

# --- generate module class --------------------------------------------------
# Pass a lowercase type-id string, then the functions.
BasicScorer = generate_from_functions('basic_scorer', score, risk_flag)

# --- instantiate (name= is required) ----------------------------------------
scorer = BasicScorer(name='my_scorer')

print('Module type identifier:', scorer.type)
print('Module name:', scorer.name)

In [ ]:
# --- run on a DataFrame -----------------------------------------------------
df = pl.DataFrame({
    'customer_id': ['C1', 'C2', 'C3', 'C4'],
    'amount':      [10.0, 50.0, 90.0, 120.0],
    'rate':        [0.5,  1.2,  1.0,  0.8],
})

result = scorer({'input': df})
print(result)

Notice that `score` and `risk_flag` are added as new columns alongside the original data. The module returns a Polars `DataFrame` (or `LazyFrame` depending on the executor — call `.collect()` if needed).

## Dependency Wiring

If one function's parameter name matches another function's `__name__`, Decider automatically injects the upstream expression — no manual wiring required.

In [ ]:
def score(amount: pl.Expr, rate: pl.Expr) -> pl.Expr:
    return amount * rate

def risk_flag(score: pl.Expr) -> pl.Expr:
    return score > pl.lit(80.0)

# --- third function depends on 'score' (sibling output) ---------------------
def score_normalised(score: pl.Expr) -> pl.Expr:
    """Normalise score to [0, 1] using a fixed max of 200."""
    return score / pl.lit(200.0)

# Decider resolves the 'score' parameter in risk_flag and score_normalised
# to the output of the 'score' function — automatically.
ScorerV2 = generate_from_functions(
    'scorer_v2',
    score,
    risk_flag,
    score_normalised,
)
scorer_v2 = ScorerV2(name='scorer_v2')

result_v2 = scorer_v2({'input': df})
print(result_v2.select(['customer_id', 'score', 'risk_flag', 'score_normalised']))

## Config Injection

Add a parameter named `config` annotated with a Pydantic model. Decider promotes all config fields onto the module itself, so you set them at instantiation time and they are injected automatically at call time.

In [ ]:
from pydantic import BaseModel

class PremiumConfig(BaseModel):
    base_rate:  float = 1.0
    multiplier: float = 2.0

def score(amount: pl.Expr, rate: pl.Expr) -> pl.Expr:
    return amount * rate

def premium_score(score: pl.Expr, config: PremiumConfig) -> pl.Expr:
    """Scale score by a configurable multiplier on top of a base rate."""
    return score * pl.lit(config.base_rate) * pl.lit(config.multiplier)

PremiumScorer = generate_from_functions('premium_scorer', score, premium_score)

# Config fields (base_rate, multiplier) appear directly on the module
m = PremiumScorer(name='premium', base_rate=1.5, multiplier=3.0)

print('base_rate  :', m.base_rate)
print('multiplier :', m.multiplier)
print('Model fields:', list(m.model_fields.keys()))

In [ ]:
result_premium = m({'input': df})
print(result_premium.select(['customer_id', 'amount', 'rate', 'score', 'premium_score']))